# L13c: Decoder-Only Transformers (GPT-style)
In this lecture, we build the _decoder-only_ transformer architecture used by GPT-style language models. We start from the transformer block introduced in L13a and add the two pieces that turn it into a generative language model: a _causal_ (lower-triangular) attention mask that prevents each position from attending to future tokens, and a language-modeling head that produces a probability distribution over the vocabulary at every position. We then assemble everything into a small decoder-only model that can be trained on raw text by next-token prediction.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Explain the role of the causal mask in autoregressive language modeling:__ Describe how the lower-triangular mask enforces the autoregressive factorization $p(x_{1},\ldots,x_{T}) = \prod_{t} p(x_{t}\mid x_{<t})$ and why this lets the model be trained on all positions of a sequence in parallel.
> * __Assemble a decoder-only language model from a transformer block:__ Combine token embeddings, positional embeddings, $L$ stacked decoder blocks, a final LayerNorm, and a linear LM head into a model that maps a sequence of token ids to a sequence of next-token logit distributions, and compute the parameter count.
> * __Sample from a trained language model:__ Apply greedy, temperature, and top-$k$ sampling to autoregressively extend a prompt, and explain the trade-off each strategy makes between repetition and diversity.

Let's get started!
___

## Example
Today, we will use the following notebook to illustrate key concepts:

> [▶ NanoGPT: a Tiny Decoder-Only LM on Tiny Shakespeare](CHEME-5820-L13c-Example-NanoGPT-Shakespeare-Spring-2026.ipynb). In this example, we train a small character-level decoder-only language model (~110k parameters, two transformer blocks, four heads) on the Tiny Shakespeare corpus and sample from it with greedy, temperature, and top-$k$ decoding. We also visualize the causal attention pattern.

___

## Recap: The Transformer Block from L13a
In the [L13a lecture](CHEME-5820-L13a-Lecture-Spring-2026.ipynb), we built the _transformer block_: a multi-head self-attention sublayer followed by a position-wise feedforward sublayer, each wrapped in a residual connection and a layer normalization. With input $\mathbf{X}\in\mathbb{R}^{n\times d}$, the (pre-norm) block computes:

> __Transformer Block (recap from L13a, pre-norm)__
>
> $$
\boxed{
\begin{align*}
\mathbf{Y} &= \mathbf{X} + \operatorname{MultiHead}(\operatorname{LayerNorm}(\mathbf{X})) \\
\mathbf{Z} &= \mathbf{Y} + \operatorname{FFN}(\operatorname{LayerNorm}(\mathbf{Y}))
\end{align*}}
> $$
> where multi-head attention is $H$ parallel scaled dot-product attention layers concatenated and projected back to dimension $d$, and the position-wise feedforward sublayer applies the same two-layer MLP independently to every row of its input.

The L13a block uses the standard self-attention formula
$$
\operatorname{Attention}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \operatorname{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^{\top}}{\sqrt{d_{k}}}\right)\mathbf{V},
$$
in which every row of $\mathbf{Q}\mathbf{K}^{\top}$ is allowed to attend to every column. Today we make exactly one change to that equation, to turn the block into a building block for autoregressive language modeling.

### A Brief Aside: Cross-Attention and Encoder-Decoder Models
Decoder-only transformers are not the only flavor in the wild. The [original transformer of Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762) has both an _encoder_ and a _decoder_, and the decoder uses a third attention sublayer called _cross-attention_ in which queries come from the decoder's hidden states but keys and values come from the encoder's output. This is what enables the encoder-decoder architecture used for tasks like machine translation, where the decoder needs to attend to the entire input sentence at every step. We will not need cross-attention for the GPT-style decoder-only model, so we mention it only for context. The interested reader is referred to the original paper for the encoder-decoder formulation.
___

## Causal Self-Attention
We want a model that, given the first $t-1$ tokens of a sequence, predicts the $t$-th token. For this to make sense at training time, the model's output at position $t$ must not depend on tokens at positions $t+1, t+2, \ldots, T$, otherwise the prediction would trivially see its own answer. Self-attention as written in L13a violates this requirement: every query position attends to every key position, including future ones.

The fix is to add a _causal mask_ that sets the attention scores at all illegal (future) positions to $-\infty$ before the softmax, so those positions receive zero attention weight.

> __Scaled Dot-Product Causal Self-Attention__
>
> Let $\mathbf{Q}, \mathbf{K}\in\mathbb{R}^{n\times d_{k}}$ and $\mathbf{V}\in\mathbb{R}^{n\times d_{v}}$ be query, key, and value matrices for a sequence of length $n$. Define the causal mask $\mathbf{M}\in\mathbb{R}^{n\times n}$ component-wise as
> $$
\mathbf{M}_{ij} = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}
> $$
> Then causal self-attention is
> $$
\boxed{
\operatorname{CausalAttention}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \operatorname{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^{\top}}{\sqrt{d_{k}}} + \mathbf{M}\right)\mathbf{V}.
}
> $$
> Because $\exp(-\infty) = 0$, every entry $(i, j)$ with $j > i$ receives zero softmax weight, so the output row $i$ is a weighted average over only the value rows at positions $1, 2, \ldots, i$.

Visually, the mask $\mathbf{M}$ looks like this for $n = 6$:

> __Causal mask shape ($n = 6$, $0$ shown for valid entries, $\bullet$ for $-\infty$)__
>
> $$
\mathbf{M} = \begin{pmatrix}
0 & \bullet & \bullet & \bullet & \bullet & \bullet \\
0 & 0 & \bullet & \bullet & \bullet & \bullet \\
0 & 0 & 0 & \bullet & \bullet & \bullet \\
0 & 0 & 0 & 0 & \bullet & \bullet \\
0 & 0 & 0 & 0 & 0 & \bullet \\
0 & 0 & 0 & 0 & 0 & 0
\end{pmatrix}
> $$

After softmax, the attention matrix has a strict lower-triangular pattern: row 1 attends only to position 1, row 2 attends to positions 1 and 2, and so on. Row $i$ has exactly $i$ non-zero entries that sum to one.

### Why This Implements an Autoregressive Factorization
Probabilistic language models factorize the joint distribution over a sequence by the chain rule:
$$
p(x_{1}, x_{2}, \ldots, x_{T}) = \prod_{t=1}^{T} p(x_{t}\mid x_{<t})\quad\text{where } x_{<t} = (x_{1}, \ldots, x_{t-1}).
$$
A decoder-only language model parameterizes each conditional $p(x_{t}\mid x_{<t})$ by a neural network. The crucial constraint is that the prediction at position $t$ must depend _only_ on tokens at positions $1, 2, \ldots, t-1$ (and possibly $t$ itself, depending on convention). The causal mask enforces exactly this constraint at every layer of the network: information at position $j > i$ cannot flow into position $i$ through any attention sublayer, so the model's output at position $i$ is a function of $x_{1}, \ldots, x_{i}$ only.

> __The big payoff: parallel training.__
>
> Because the causal mask makes each output position depend only on its own past, _all_ positions of a sequence can be processed by the network in a single forward pass and trained simultaneously by gradient descent. Compare this to a recurrent language model, which has to step through positions one at a time during both forward and backward passes. The causal mask is what makes the GPU-friendly parallelism of self-attention compatible with the inherently sequential structure of language.
___

## The Decoder-Only Block
A _decoder-only block_ is the L13a transformer block with the multi-head self-attention sublayer replaced by its causal version.

> __Decoder-Only Block__
>
> Let $\mathbf{X}\in\mathbb{R}^{n\times d}$ be the input. The block computes:
> $$
\boxed{
\begin{align*}
\mathbf{Y} &= \mathbf{X} + \operatorname{CausalMultiHead}(\operatorname{LayerNorm}(\mathbf{X})) \\
\mathbf{Z} &= \mathbf{Y} + \operatorname{FFN}(\operatorname{LayerNorm}(\mathbf{Y}))
\end{align*}}
> $$
> where $\operatorname{CausalMultiHead}$ is multi-head attention with the causal mask applied to every head, and $\operatorname{FFN}$ is the same position-wise two-layer MLP from L13a. The residual connections and layer normalizations play exactly the same role as in L13a: the residuals provide gradient highways and let deep stacks be trainable, and the layer norms keep activations on a stable scale.

The block is otherwise structurally identical to L13a's transformer block, including the parameter count: $\sim 12 d^{2}$ in the dominant terms ($4 d^{2}$ for the four attention projection matrices and $8 d^{2}$ for the two feedforward weight matrices), plus linear-in-$d$ contributions from biases and layer norm parameters.
___

## Assembling a Decoder-Only Language Model
A complete GPT-style language model wraps a stack of $L$ decoder-only blocks with three more pieces: a token embedding lookup, a positional embedding, and a final language-modeling head.

> __Decoder-Only Language Model__
>
> Let $V$ be the vocabulary size, $d$ the model dimension, $L$ the number of stacked blocks, and $n_{\max}$ the maximum context length. Define
> * $\mathbf{E}_{\text{tok}}\in\mathbb{R}^{V\times d}$, the token embedding matrix
> * $\mathbf{E}_{\text{pos}}\in\mathbb{R}^{n_{\max}\times d}$, the positional embedding matrix
> * $L$ decoder-only blocks $\mathcal{B}_{1}, \mathcal{B}_{2}, \ldots, \mathcal{B}_{L}$
> * a final LayerNorm operating on the $d$-dimensional embedding axis
> * a language-modeling head $\mathbf{W}_{\text{LM}}\in\mathbb{R}^{V\times d}$, a linear projection to vocabulary logits
>
> Given a sequence of token ids $\mathbf{x} = (x_{1}, x_{2}, \ldots, x_{T})$ with $T \leq n_{\max}$, the model computes:
> $$
\boxed{
\begin{align*}
\mathbf{H}^{(0)} &= \mathbf{E}_{\text{tok}}[\mathbf{x}, :] + \mathbf{E}_{\text{pos}}[1:T, :] \quad\in\mathbb{R}^{T\times d} \\
\mathbf{H}^{(\ell)} &= \mathcal{B}_{\ell}(\mathbf{H}^{(\ell-1)}),\quad \ell = 1, 2, \ldots, L \\
\tilde{\mathbf{H}} &= \operatorname{LayerNorm}(\mathbf{H}^{(L)}) \\
\boldsymbol{\ell} &= \tilde{\mathbf{H}}\,\mathbf{W}_{\text{LM}}^{\top}\quad\in\mathbb{R}^{T\times V}
\end{align*}}
> $$
> Row $t$ of $\boldsymbol{\ell}$ is a $V$-dimensional vector of unnormalized logits for the next-token distribution conditional on $x_{1}, \ldots, x_{t}$. Applying a softmax to row $t$ gives $p(x_{t+1}\mid x_{\leq t})$.

The $\mathbf{E}_{\text{tok}}[\mathbf{x}, :]$ notation is shorthand for "look up rows $x_{1}, x_{2}, \ldots, x_{T}$ of the token embedding matrix and stack them," and is implemented as an embedding-lookup (a single fancy-indexing operation) at the start of the forward pass.
___

## Training: Next-Token Prediction
Training a decoder-only language model is _next-token prediction_: given a sequence of $T+1$ tokens, the model is shown the first $T$ tokens as input and asked to predict the next token at every position. The targets $\mathbf{y}$ are simply the input shifted by one position: $y_{t} = x_{t+1}$.

> __Training loss__
>
> Let $\mathbf{x} = (x_{1}, \ldots, x_{T})$ and $\mathbf{y} = (x_{2}, \ldots, x_{T+1})$ be an input/target pair from a corpus, and let the model produce logits $\boldsymbol{\ell}\in\mathbb{R}^{T\times V}$ as above. The cross-entropy loss is
> $$
\boxed{
\mathcal{L}(\theta;\, \mathbf{x},\mathbf{y}) = -\frac{1}{T}\sum_{t=1}^{T}\log p_{\theta}(y_{t}\mid x_{\leq t}) = -\frac{1}{T}\sum_{t=1}^{T}\log\operatorname{softmax}(\boldsymbol{\ell}_{t,:})_{y_{t}}.
}
> $$
> Averaging over a batch of $B$ sequences and over all $T$ positions gives the per-token cross-entropy used to take optimizer steps.

> __Why all $T$ positions can be trained in parallel.__
>
> Because of the causal mask, the prediction $p_{\theta}(y_{t}\mid x_{\leq t})$ depends only on tokens at positions $1, \ldots, t$, even though the model was given the entire sequence $\mathbf{x}$ as input. So a single forward pass over $\mathbf{x}$ produces $T$ valid (input, target) pairs at once and the loss summed over $t$ trains the model on all of them. This is sometimes called _teacher forcing_: at training time, position $t$ is conditioned on the _true_ tokens at positions $1, \ldots, t-1$ rather than on the model's own (potentially wrong) past predictions. Teacher forcing combined with the causal mask is what makes a single batched forward pass compute $T$ training signals at once, instead of $T$ sequential forward passes through a recurrent model.
___

## Inference: Autoregressive Sampling
At inference time, the model has no future tokens to feed itself; it has to generate them one at a time. The standard generation loop is:

> __Autoregressive generation loop__
>
> Given a prompt $\mathbf{x} = (x_{1}, \ldots, x_{T_{0}})$ and a target generation length $N$:
> 1. For $t = T_{0} + 1, T_{0} + 2, \ldots, T_{0} + N$:
>    1. Form the model input from the most recent $\min(t - 1,\, n_{\max})$ tokens of the running sequence.
>    2. Run a single forward pass through the model to obtain logits $\boldsymbol{\ell}\in\mathbb{R}^{T\times V}$, where $T$ is the input length.
>    3. Take the _last row_ of $\boldsymbol{\ell}$, which is the logit vector for position $t$.
>    4. Sample one token $x_{t}$ from a distribution derived from this logit vector (see below).
>    5. Append $x_{t}$ to the running sequence and continue.

There are three standard ways to convert the logit vector at the last position into a sampled token. Each one trades off determinism against diversity in different ways.

> __Three sampling strategies__
>
> * __Greedy decoding ($\operatorname{argmax}$):__ pick the token with the largest logit. This is deterministic and reproducible, but tends to fall into repetition loops because the highest-probability continuation of "the the the" is often "the" again.
> * __Temperature sampling:__ scale the logits by $1/\tau$ for $\tau > 0$, apply softmax, and sample. Small $\tau$ (close to $0$) approaches greedy decoding; $\tau = 1$ samples from the model's natural distribution; large $\tau$ (greater than $1$) flattens the distribution and produces more random outputs.
> * __Top-$k$ sampling:__ before applying softmax, set all but the $k$ largest logits to $-\infty$, so sampling is restricted to the $k$ most likely tokens. This avoids occasional very-low-probability "junk" tokens that pure temperature sampling can produce, while still keeping diversity within the high-probability region.

In practice, temperature sampling at $\tau\approx 0.8$ combined with top-$k$ sampling at $k\approx 40$ is a reasonable default for character-level and small word-level models.

___

## Parameter Count
The total parameter count of a decoder-only model is dominated by the $L$ decoder blocks plus the embedding tables.

> __Parameter Count: Decoder-Only Language Model__
>
> A decoder-only language model with vocabulary size $V$, model dimension $d$, $L$ stacked blocks, $H$ attention heads (with $d_{k} = d_{v} = d/H$), maximum context length $n_{\max}$, and feedforward dimension $d_{ff} = 4d$ contains:
> * _Token embedding_: $V\,d$ parameters
> * _Positional embedding_: $n_{\max}\,d$ parameters
> * _$L$ decoder blocks_: $L\,(12\,d^{2} + 9\,d)$ parameters in total (the $12 d^{2}$ dominant term plus linear-in-$d$ biases and LayerNorm parameters)
> * _Final LayerNorm_: $2\,d$ parameters
> * _LM head_: $V\,d$ parameters (no bias)
>
> The total parameter count is approximately
> $$
\begin{align*}
N_{\text{total}} \approx 2\,V\,d + n_{\max}\,d + L\cdot 12\,d^{2}\quad\blacksquare
\end{align*}
> $$
> Two terms dominate at different scales: the embedding terms scale linearly in $V\,d$ and dominate when $V$ is much larger than $L\,d$ (a small model with a large vocabulary), while the decoder-stack term scales as $L\,d^{2}$ and dominates when the model is deep and wide (a large model with a small vocabulary).

#### Numerical Example: Tiny Shakespeare NanoGPT
For the example notebook, we use a character-level NanoGPT with $V = 65$, $d = 64$, $H = 4$, $L = 2$, $n_{\max} = 64$, and $d_{ff} = 256$:
* Token embedding: $65\cdot 64 = 4{,}160$
* Positional embedding: $64\cdot 64 = 4{,}096$
* Per block: $12\cdot 64^{2} + 9\cdot 64 = 49{,}152 + 576 = 49{,}728$. Two blocks: $99{,}456$.
* Final LayerNorm: $2\cdot 64 = 128$
* LM head: $65\cdot 64 = 4{,}160$
* **Total: $112{,}000$ parameters.**

This is small enough to train on CPU in a few minutes and to ship as a checkpoint with the example notebook. By comparison, a single transformer block at $d = 128$ from the L13a numerical example had $197{,}760$ parameters, and modern GPT-style models are billions of parameters at $d \in \{1024, 4096, 12288\}$ with $L \in \{12, 24, 96, \ldots\}$.

___

## How Decoder-Only Compares to Other Transformer Variants
Decoder-only is one of three major transformer families. The table summarizes the high-level differences.

> __Transformer architecture variants__
>
> | Property | Encoder-Only (BERT) | Encoder-Decoder (T5, original Vaswani) | Decoder-Only (GPT) |
> |---|---|---|---|
> | Self-attention mask | None (full bidirectional) | Encoder: none. Decoder: causal | Causal |
> | Cross-attention sublayer | No | Yes (decoder attends to encoder output) | No |
> | Typical training objective | Masked language modeling | Sequence-to-sequence (translation, summarization) | Next-token prediction |
> | Generation? | No (used for classification/regression) | Yes (with the decoder) | Yes |
> | Example uses | Sentence classification, NER | Translation, summarization, instruction following | Text generation, code generation, chat |

We focus on the decoder-only family in this lecture because it is the simplest of the three (a single stack with one type of attention sublayer) and because it underpins most modern large language models. The encoder-only and encoder-decoder variants share most of the same building blocks; their differences are primarily in which attention masks are used where and which heads are attached to the output.

___

## Summary
A decoder-only transformer is the L13a transformer block with the self-attention sublayer made causal, stacked $L$ times, and wrapped with token and positional embeddings on the input side and a linear language-modeling head on the output side. The causal mask makes the model autoregressive, which lets it be trained by next-token prediction on raw text and sampled from autoregressively at inference time.

> __Key Takeaways:__
>
> * **The causal mask is the only architectural change.** Replacing self-attention with causal self-attention turns the L13a transformer block into a decoder-only block. Everything else, including residuals, layer normalization, multi-head attention, and the position-wise feedforward sublayer, is reused unchanged.
> * **Causal masking enables parallel training of an inherently sequential model.** Because each output position depends only on its own past, all positions of a training sequence can be predicted in a single forward pass and trained simultaneously by cross-entropy on the shifted targets. This is the property that lets transformers train efficiently on long sequences in a way that recurrent models cannot.
> * **Sampling strategy controls the diversity-coherence trade-off.** Greedy decoding is reproducible but degenerates into repetition; temperature sampling restores diversity at the cost of occasional incoherent tokens; top-$k$ sampling combines the two by clipping the tail of the distribution. All three are easy to implement on top of the same trained model.

For an applied example training a small decoder-only model on Tiny Shakespeare and sampling from it with all three strategies, see the [L13c example notebook](CHEME-5820-L13c-Example-NanoGPT-Shakespeare-Spring-2026.ipynb).
___